In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# ==========================================================
# Configuración del navegador
# ==========================================================
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ==========================================================
# Variables
# ==========================================================
datos_totales = []
paginas_objetivo = 5

try:
    driver.get("https://books.toscrape.com/")

    # ==========================================================
    # Recorrer varias páginas
    # ==========================================================
    for i in range(paginas_objetivo):
        print(f"--- Procesando página {i + 1} ---")

        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CLASS_NAME, "product_pod")
            )
        )

        productos = driver.find_elements(By.CLASS_NAME, "product_pod")

        for p in productos:
            try:
                precio = p.find_element(
                    By.CSS_SELECTOR,
                    "p.price_color"
                ).text

                estrellas = p.find_element(
                    By.CSS_SELECTOR,
                    "p.star-rating"
                ).get_attribute("class")

                estrellas = estrellas.replace("star-rating ", "")

                datos_totales.append({
                    "precio": precio,
                    "estrellas": estrellas,
                    "pagina": i + 1
                })

            except Exception:
                continue

        # ==========================================================
        # Ir a la siguiente página
        # ==========================================================
        try:
            boton_siguiente = driver.find_element(
                By.XPATH,
                "//li[@class='next']/a"
            )

            boton_siguiente.click()

            time.sleep(2)

        except Exception:
            print("No existe una página siguiente.")
            break

    # ==========================================================
    # Guardar resultados
    # ==========================================================
    df = pd.DataFrame(datos_totales)

    df.to_csv(
        "datos_paginados.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")
    print(df.head())

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    driver.quit()

--- Procesando página 1 ---
--- Procesando página 2 ---
--- Procesando página 3 ---
--- Procesando página 4 ---
--- Procesando página 5 ---

Proceso finalizado con éxito.
Total de registros capturados: 100
   precio estrellas  pagina
0  £51.77     Three       1
1  £53.74       One       1
2  £50.10       One       1
3  £47.82      Four       1
4  £54.23      Five       1
